# Kaggriculture Stage 3 v5: Counterfactual Market Search
Проверяет конкретные правила продажи в одинаковых играх, отбирает лучшие на отдельных контрольных сидах и сохраняет только доказанное улучшение агента 1601.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive подключен до начала долгой работы.')

In [ ]:
import os, subprocess
from pathlib import Path
PROJECT = Path('/content/Kaggriculture')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/GrigoriiIurev/Kaggriculture.git', str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(PROJECT)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Зависимости установлены.')

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Kaggriculture')
INCUMBENT = DRIVE_ROOT / 'results/rl_boatlee_market_v2/submission.tar.gz'
CANDIDATES = 12
FINALISTS = 2
SCREEN_OPPONENTS = 2
SCREEN_SEED_COUNT = 1
FINAL_SEED_COUNT = 3
SUBMIT_TO_KAGGLE = False
SUBMIT_UNPROMOTED = False
SUBMISSION_MESSAGE = 'Stage 3 counterfactual market search v5'
POOL = DRIVE_ROOT / 'league/opponent_pool.json'
assert INCUMBENT.is_file(), f'Не найден агент 1601: {INCUMBENT}'
assert POOL.is_file(), f'Сначала завершите Stage 2: {POOL}'
print('Incumbent:', INCUMBENT)
print('Opponent pool:', POOL)

In [ ]:
import subprocess, sys
LOG = DRIVE_ROOT / 'results/stage3_counterfactual_market_v5/pipeline.log'
LOG.parent.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', 'run_counterfactual_market_pipeline.py',
    '--drive-root', str(DRIVE_ROOT),
    '--incumbent', str(INCUMBENT),
    '--candidates', str(CANDIDATES),
    '--finalists', str(FINALISTS),
    '--screen-opponents', str(SCREEN_OPPONENTS),
    '--screen-seed-count', str(SCREEN_SEED_COUNT),
    '--final-seed-count', str(FINAL_SEED_COUNT),
    '--message', SUBMISSION_MESSAGE,
]
if SUBMIT_TO_KAGGLE:
    command.append('--submit')
if SUBMIT_UNPROMOTED:
    command.append('--submit-unpromoted')
print('Запускаю:', ' '.join(command), flush=True)
print('Полный лог:', LOG, flush=True)
with LOG.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line); log.flush()
    code = process.wait()
if code:
    raise RuntimeError(f'Pipeline завершился с кодом {code}. Последние строки находятся в {LOG}')
print('Counterfactual search завершён без ошибок.', flush=True)

In [ ]:
import json, tarfile
RESULT = DRIVE_ROOT / 'results/stage3_counterfactual_market_v5'
report = json.loads((RESULT / 'counterfactual_search_report.json').read_text())
receipt = json.loads((RESULT / 'submission_receipt.json').read_text())
print('Новая модель прошла проверку:', report['promoted'])
print('Архив:', receipt['output'])
print('Размер:', receipt['bytes'], 'байт')
with tarfile.open(RESULT / 'submission.tar.gz', 'r:gz') as archive:
    print('main.py в корне:', 'main.py' in archive.getnames())